# Sentiment Analysis pada Aplikasi Fatsecret

Fatsecret adalah salah satu aplikasi kalori counter dan pelacak diet sehat yang dapat diunduh di Google PlayStore. Aplikasi ini sangat membantu bagi mereka yang hendak melakukan pemantauan asupan harian karena terdapat kalori counter. User dapat memasukkan daftar makanan harian dan menghitung kalori asupan hariannya. Selain itu, terdapat fitur-fitur pemantauan BMI, berat badan, aktivitas harian, program diet lengkap, bahkan fitur sosial media khusus diet.

Sentimen analisis ini akan mengulas bagaimana tanggapan user terhadap aplikasi tersebut. Sentimen di dapat dari crawling reviews di Google Play Store. <br>

**Model Ekstraksi Fitur :**
1. TF-IDF
2. BoW
<br>
**Model Machine Learning :**
1. Random Forest
2. Decision Tree
3. Logistic Regression
4. Naive Bayes
5. XGBoost
<br>

Untuk metode splitting menggunakan Cross Validation dengan K = 5. Cross Validation akan membagi dataset dalam 5 fold dan mengiterasi semua data. Hal ini bertujuan agar semua data bisa merasakan menjadi train dataset maupun test dataset hingga didapatkan akurasi testing paling maksimal.

# Import Library

In [1]:
# KEBUTUHAN UMUM
import pandas as pd
import numpy as np
import pickle

# KEBUTUHAN PREPROCESSING TEKS
import re
import string
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
from nltk.tokenize import  word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# KEBUTUHAN LABELING
import csv
import requests
from io import StringIO

# KEBUTUHAN VISUALISASI DAN EKSPLORASI LABEL
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import  WordCloud

# KEBUTUUHAN FITUR EXTRACTION DAN MODELLING
from sklearn.model_selection import cross_val_score, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import  LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.svm import SVC
from xgboost import  XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Exploratory Data Analysis

In [2]:
df = pd.read_csv("fatsecret_scraped.csv")
df.head()

,userName,score,at,content
0,Pengguna Google,5,2025-09-25 09:50:09,sangat membantu mengontrol makanan harian
1,Pengguna Google,5,2025-09-25 06:01:24,Sangat membantu...!
2,Pengguna Google,5,2025-09-20 06:00:53,sangat membantu
3,Pengguna Google,5,2025-09-19 05:57:22,Terima kasih. sangat membantu
4,Pengguna Google,5,2025-09-18 20:45:47,"sangatlah baguss, saya bis akonsisten memakai ini"


In [3]:
# CEK INFORMASI DATAFRAME
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8240 entries, 0 to 8239
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   userName  8240 non-null   object
 1   score     8240 non-null   int64 
 2   at        8240 non-null   object
 3   content   8240 non-null   object
dtypes: int64(1), object(3)
memory usage: 257.6+ KB


In [4]:
df.isnull().sum()

userName    0
score       0
at          0
content     0
dtype: int64

In [5]:
df_cleaned = df.dropna()

In [6]:
df_cleaned.isnull().sum()

userName    0
score       0
at          0
content     0
dtype: int64

In [7]:
# CEK DUPLIKASI
duplicate = df_cleaned.duplicated().sum()
print("Baris terduplikasi: ", duplicate)

Baris terduplikasi:  0


In [8]:
# CEK DIMENSI DATA
df_cleaned.shape

(8240, 4)

# Preprocessing Text

### Fungsi Preprocessing

In [9]:
# FUNGSI CLEANING TEXT
def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text) #hapus mention
    text = re.sub(r'#[A-Za-z0-9]+', '', text) #hapus hashtag
    text = re.sub(r'RT[\s]', '', text) #hapus RT
    text = re.sub(r'RW[\s]', '', text) #hapus RW
    text = re.sub(r"http\S+", '', text) #hapus link
    text = re.sub(r'[0-9]+', '', text) #hapus angka
    text = re.sub(r'[^\w\s]', '', text) #hapus karakter selain huruf dan angka
    text = text.replace('\n', ' ') #ganti baris baru dengan spasi
    text = text.translate(str.maketrans('', '', string.punctuation)) #hapus semua tanda baca
    text = text.strip(' ') #hapus karakter spasi dari kiri dan kanan teks
    return text

# FUNGSI CASE FOLDING
def caseFolding(text):
    text = text.lower()
    return text

# FUNGSI HANDLING SLANK
slangwords = {"@": "di", "abis": "habis", "wtb": "beli", "masi": "masih", "wts": "jual", "wtt": "tukar", "bgt": "banget", "maks": "maksimal", "d": "di", "gak": "tidak", "medsos": "media sosial", "pd": "pada", "ga": "tidak", "mslh": "masalah", "santuy": "santai", "trus": "terus", "yg": "yang", "oke": "baik", "kasi": "kasih", "gw": "saya", "wapres": "wakil presiden", "kepingin": "ingin", "auto": "otomatis", "utk": "untuk", "bbrp" : "beberapa", "dri": "dari", "org2": "orang-orang", "ancur": "hancur", "spt": "seperti", "bgt": "banget", "btw": "ngomong-ngomong", "anjir": "kaget", "anjay": "takjub","busyeet": "busyet","dach": "deh","ga": "tidak","gak": "tidak","gpp": "tidak apa-apa","gw": "saya","loe": "kamu","lu": "kamu","maen": "main","nggak": "tidak","ngga": "tidak","nyimak": "menyimak","sumpah": "serius","wkwk": "tertawa","wk": "tertawa","yoi": "iya","dgn": "dengan","sampe": "sampai","tdk": "tidak","kalo": "kalau","om": "paman","loh": "ekspresi heran","yaaa": "ya","org": "orang",}
def fix_slangwords(text):
    words = text.split()
    fixed_words = []

    for word in words:
        if word.lower() in slangwords:
            fixed_words.append(slangwords[word.lower()])
        else:
            fixed_words.append(word)

    fixed_text = ' '.join(fixed_words)
    return fixed_text

# FUNGSI TOKENISASI
def tokenizingText(text):
    text = word_tokenize(text)
    return  text

# STOPWORDS HANDLING
def stopwordsText(text):
    listStopwords = set(stopwords.words('indonesian'))
    listStopwords1 = set(stopwords.words('english'))
    listStopwords.update(listStopwords1)
    listStopwords.update(['iya','yaa','gak','nya','na','sih','ku',"di","ga","ya","gaa","loh","kah","woi","woii","woy", 'gk', 'cuy', 'dih', 'beuh', 'eee', 'heee', 'nih', 'nihh', 'dong', 'mulu', 'mah', ])
    filtered = []
    for txt in text:
        if txt not in listStopwords:
            filtered.append(txt)
    text = filtered
    return text

# FUNGSI STEMMING TEKS
def stemmingText(text):
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()

    #Stemming setiap kata
    stemmed_words = [stemmer.stem(word) for word in text]

    #Gabung kata yang telah distem dengan spasi
    stemmed_text = ' '.join(stemmed_words)

    return stemmed_text

# SIMPAN NASKAH AKHIR
def toSentence(list_words):
    sentence = ' '.join(word for word in list_words)
    return sentence

### Implementasi Fungsi

In [10]:
 # IMPLEMENTASI FUNGSI
# Bersihkan teks dg regex
df_cleaned['text_clean'] = df_cleaned['content'].apply(cleaningText)

# Case folding
df_cleaned['text_caseFolding'] = df_cleaned['text_clean'].apply(caseFolding)

# Slang Handling
df_cleaned['text_slangWords'] = df_cleaned['text_caseFolding'].apply(fix_slangwords)

# Tokenisasi
df_cleaned['text_tokenizing'] = df_cleaned['text_slangWords'].apply(tokenizingText)

# Stopwords Handling
df_cleaned['text_stopwords'] = df_cleaned['text_tokenizing'].apply(stopwordsText)

# Join text
df_cleaned['text_akhir'] = df_cleaned['text_stopwords'].apply(toSentence)

df_cleaned.head()

,userName,score,at,content,text_clean,text_caseFolding,text_slangWords,text_tokenizing,text_stopwords,text_akhir
0,Pengguna Google,5,2025-09-25 09:50:09,sangat membantu mengontrol makanan harian,sangat membantu mengontrol makanan harian,sangat membantu mengontrol makanan harian,sangat membantu mengontrol makanan harian,"[sangat, membantu, mengontrol, makanan, harian]","[membantu, mengontrol, makanan, harian]",membantu mengontrol makanan harian
1,Pengguna Google,5,2025-09-25 06:01:24,Sangat membantu...!,Sangat membantu,sangat membantu,sangat membantu,"[sangat, membantu]",[membantu],membantu
2,Pengguna Google,5,2025-09-20 06:00:53,sangat membantu,sangat membantu,sangat membantu,sangat membantu,"[sangat, membantu]",[membantu],membantu
3,Pengguna Google,5,2025-09-19 05:57:22,Terima kasih. sangat membantu,Terima kasih sangat membantu,terima kasih sangat membantu,terima kasih sangat membantu,"[terima, kasih, sangat, membantu]","[terima, kasih, membantu]",terima kasih membantu
4,Pengguna Google,5,2025-09-18 20:45:47,"sangatlah baguss, saya bis akonsisten memakai ini",sangatlah baguss saya bis akonsisten memakai ini,sangatlah baguss saya bis akonsisten memakai ini,sangatlah baguss saya bis akonsisten memakai ini,"[sangatlah, baguss, saya, bis, akonsisten, mem...","[baguss, bis, akonsisten, memakai]",baguss bis akonsisten memakai


In [11]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8240 entries, 0 to 8239
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   userName          8240 non-null   object
 1   score             8240 non-null   int64 
 2   at                8240 non-null   object
 3   content           8240 non-null   object
 4   text_clean        8240 non-null   object
 5   text_caseFolding  8240 non-null   object
 6   text_slangWords   8240 non-null   object
 7   text_tokenizing   8240 non-null   object
 8   text_stopwords    8240 non-null   object
 9   text_akhir        8240 non-null   object
dtypes: int64(1), object(9)
memory usage: 643.9+ KB


# Labeling Text

In [58]:
# KATA-KATA POSITIF
lexicon_positive = dict()
response = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_positive.csv') # File csv dari GitHub yang isinya daftar kata positif

# Menambahkan beberapa lexicon positif
new_positive_words = {
    'bantu konsisten': 3,
    'hebat': 2,
    'sangat membantu' : 2,
    'membantu': 2,
    'tidak mengecewakan' : 2,
    'bagus banget' : 3,
    'kalori terpantau' : 2,
    'menu lengkap' : 2,
    'bantu menurunkan' : 3,
    'mantap': 1,
    'keren banget' : 3,
    'aku suka' : 2,
    'bagus banget':3
}

# Gabungkan kamus
lexicon_positive.update(new_positive_words)

if response.status_code==200:
    reader = csv.reader(StringIO(response.text), delimiter=',')

    for row in reader:
        lexicon_positive[row[0]] = int(row[1]) # Tambahkan daftar kata positif dan skor ke kamus
else:
    print("Failed to fetch")

# KATA-KATA NEGATIF
lexicon_negative = dict()
response = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_negative.csv') # File csv dari GitHub yang isinya daftar kata negatif

# Menambahkan beberapa lexicon positif
new_negative_words = {
    'mengecewakan':-2,
    'sangat mengecewakan': -3,
    'lemot': -1,
    'bug':-2,
    'tidak lengkap':-2,
    'tidak komplit':-2,
    'nggak bisa login':-2,
    'burik':-1,
    'jelek banget':-3,
    'loading lama': -2,
    'bagus':-3
}

# Gabungkan kamus
lexicon_negative.update(new_negative_words)

if response.status_code==200:
    reader = csv.reader(StringIO(response.text), delimiter=',')

    for row in reader:
        lexicon_negative[row[0]] = int(row[1]) # Tambahkan daftar kata negatif dan skor ke kamus
else:
    print("Failed to fetch")

In [59]:
# TENTUKAN POLARITAS (KE POSITIF ATAU NEGATIF)

def sentiment_analysis_lexicon(text):
    score = 0

    for word in text:
        if(word in lexicon_positive): # Untuk lexicon positif
            score = score + lexicon_positive[word]

    for word in text:
        if(word in lexicon_negative): # Untuk lexicon negatif
            score = score + lexicon_negative[word]

    polarity='' # inisialisasi variabel polaritas

    if (score > 0):
        polarity = 'Positive'
    elif(score < 0):
        polarity = 'Negative'
    else:
        polarity = 'Neutral'

    return score, polarity

results = df_cleaned['text_stopwords'].apply(sentiment_analysis_lexicon)
results = list(zip(*results))
df_cleaned['polarity_score'] = results[0]
df_cleaned['sentiment'] = results[1]
print(df_cleaned['sentiment'].value_counts())

sentiment
Positive    3016
Negative    2927
Neutral     2297
Name: count, dtype: int64


In [14]:
# SIMPAN HASIL POLARITAS
df_cleaned.to_csv("Fatsecret-Polarity.csv")

# Ekstraksi Fitur

In [15]:
X = df_cleaned['text_akhir']
y = df_cleaned['sentiment']

le = LabelEncoder()
y_encoded = le.fit_transform(y)
le.classes_

array(['Negative', 'Neutral', 'Positive'], dtype=object)

0 = Negative <br>
1 = Neutral <br>
2 = Positive

In [16]:
# TF-IDF untuk seluruh data
tfidf = TfidfVectorizer(max_features=250, min_df=20, max_df=0.8)
X_tfidf = tfidf.fit_transform(X)

In [17]:
# BoW untuk seluruh data
bow = CountVectorizer()
X_bow = bow.fit_transform(X)

# Model Building

In [18]:
# Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True,random_state=42)

In [19]:
def eval_model_cv(model, X, y, cv_splits=5):
    cv = StratifiedKFold(n_splits=5, shuffle=True,random_state=42)

    y_pred = cross_val_predict(model, X, y, cv=cv)

    accuracy = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred, average='weighted')
    recall = recall_score(y, y_pred, average='weighted')
    f1 = f1_score(y, y_pred, average='weighted')

    # Print hasil
    print(f"========= Evaluasi Model {model} ==========")
    print(f"Accuracy train: {accuracy:.4f}")
    print(f"Accuracy test: {accuracy}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1_Score: {f1:4f}")
    print("==================================================\n")



### Random Forest

In [20]:
# TF-IDF
rf_tf_model = RandomForestClassifier(n_estimators=100, max_depth=100, class_weight='balanced', random_state=42)
eval_model_cv(rf_tf_model, X_tfidf, y_encoded, cv_splits=5)

========= Evaluasi Model RandomForestClassifier(class_weight='balanced', max_depth=100, random_state=42) ==========
Accuracy train: 0.8677
Accuracy test: 0.8677184466019418
Precision: 0.8684
Recall: 0.8677
F1_Score: 0.867770



In [21]:
# Fit model
rf_tf_model.fit(X_tfidf, y_encoded)

,n_estimators,100
,criterion,'gini'
,max_depth,100
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [22]:
# BoW
rf_bow_model = RandomForestClassifier(class_weight='balanced', random_state=42)
eval_model_cv(rf_bow_model, X_bow, y_encoded, cv_splits=5)

========= Evaluasi Model RandomForestClassifier(class_weight='balanced', random_state=42) ==========
Accuracy train: 0.8755
Accuracy test: 0.8754854368932039
Precision: 0.8765
Recall: 0.8755
F1_Score: 0.875536



In [23]:
# Fit model
rf_bow_model.fit(X_bow, y_encoded)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


### Decision Tree

In [24]:
# TF-IDF
dt_tf_model = DecisionTreeClassifier(class_weight='balanced', random_state=42)
eval_model_cv(dt_tf_model, X_tfidf, y_encoded, cv_splits=5)

========= Evaluasi Model DecisionTreeClassifier(class_weight='balanced', random_state=42) ==========
Accuracy train: 0.8396
Accuracy test: 0.8395631067961165
Precision: 0.8397
Recall: 0.8396
F1_Score: 0.839362



In [25]:
# Fit model
dt_tf_model.fit(X_tfidf, y_encoded)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,'balanced'


In [26]:
dt_bow_model = DecisionTreeClassifier(class_weight='balanced', random_state=42)
eval_model_cv(dt_bow_model, X_bow, y_encoded, cv_splits=5)

========= Evaluasi Model DecisionTreeClassifier(class_weight='balanced', random_state=42) ==========
Accuracy train: 0.8621
Accuracy test: 0.8621359223300971
Precision: 0.8626
Recall: 0.8621
F1_Score: 0.861997



In [27]:
# Fit model
dt_bow_model.fit(X_bow, y_encoded)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,'balanced'


### Logistic Regression

In [28]:
# TF-IDF
lr_tf_model = LogisticRegression(class_weight='balanced', random_state=42)
eval_model_cv(lr_tf_model, X_tfidf, y_encoded, cv_splits=5)

========= Evaluasi Model LogisticRegression(class_weight='balanced', random_state=42) ==========
Accuracy train: 0.8809
Accuracy test: 0.8809466019417476
Precision: 0.8821
Recall: 0.8809
F1_Score: 0.881191



In [29]:
# Fit model
lr_tf_model.fit(X_tfidf, y_encoded)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [30]:
# BoW
lr_bow_model = LogisticRegression(class_weight='balanced', random_state=42)
eval_model_cv(lr_bow_model, X_bow, y_encoded, cv_splits=5)

========= Evaluasi Model LogisticRegression(class_weight='balanced', random_state=42) ==========
Accuracy train: 0.8790
Accuracy test: 0.879004854368932
Precision: 0.8822
Recall: 0.8790
F1_Score: 0.879589



In [31]:
# Fit model
lr_bow_model.fit(X_bow, y_encoded)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


### SVM

In [32]:
# TF-IDF
svm_tf_model = SVC(class_weight='balanced', random_state=42)
eval_model_cv(svm_tf_model, X_tfidf, y_encoded, cv_splits=5)

========= Evaluasi Model SVC(class_weight='balanced', random_state=42) ==========
Accuracy train: 0.8840
Accuracy test: 0.8839805825242718
Precision: 0.8840
Recall: 0.8840
F1_Score: 0.883925



In [33]:
# Fit model
svm_tf_model.fit(X_tfidf, y_encoded)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,'balanced'
,verbose,False


In [34]:
# BoW
svm_bow_model = SVC(class_weight='balanced', random_state=42)
eval_model_cv(svm_bow_model, X_bow, y_encoded, cv_splits=5)

========= Evaluasi Model SVC(class_weight='balanced', random_state=42) ==========
Accuracy train: 0.8710
Accuracy test: 0.870995145631068
Precision: 0.8754
Recall: 0.8710
F1_Score: 0.871112



In [35]:
# Fit model
svm_bow_model.fit(X_bow, y_encoded)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,'balanced'
,verbose,False


### XGBoost

In [36]:
# TF-IDF
xgb_tf_model = XGBClassifier(
    max_depth=5,
    learning_rate=0.3,
    n_estimators=100,
    objective='multi:softprob'
)

eval_model_cv(xgb_tf_model, X_tfidf, y_encoded, cv_splits=5)

========= Evaluasi Model XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.3, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...) ==========
Accuracy train: 0.8723
Accuracy test: 0.8723300970873786
Precision: 0.8733
Recall: 0.8723
F1_Score: 0.872408



In [37]:
# Fit model
xgb_tf_model.fit(X_tfidf, y_encoded)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [38]:
# BoW
xgb_bow_model = XGBClassifier(
    max_depth=5,
    learning_rate=0.3,
    n_estimators=100,
    objective='multi:softprob'
)

eval_model_cv(xgb_bow_model, X_bow, y_encoded, cv_splits=5)

========= Evaluasi Model XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.3, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...) ==========
Accuracy train: 0.8725
Accuracy test: 0.8724514563106797
Precision: 0.8743
Recall: 0.8725
F1_Score: 0.872691



In [39]:
# Fit model
xgb_bow_model.fit(X_bow, y_encoded)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


## Kesimpulan

**Berikut Kesimpulan :**
Dari hasil pelatihan 5 model dengan masing2 model dipadukan dengan 2 model ekstraksi fitur, serta menggunakan cross validation untuk splitting dataset dihasilkan:

1. Random Forest (TF-IDF = 86% dan BoW = 87%)
2. Decision Tree (TF-IDF = 83% dan BoW = 85%)
3. Logistic Regression (TF-IDF = 87% dan BoW = 87%)
4. SVM (TF-IDF = 88% dan BoW = 87%)
5. XGBoost (TF-IDF = 87% dan BoW = 87%)

Semua model yang dibuild memiliki akurasi yang tinggi (> 85%) kecuali model Decision Tree yang dipadukan dengan TF-IDF. Sementara akurasi testing tertinggi dimiliki oleh SVM dengan model ekstraksi fitur TF-IDF

# Inference Model

In [79]:
test = [
    "susah login. jelek banget nih aplikasi",
    "Mantap, aplikasinya sangat membantu",
    "fitur masih kurang lengkap",
    "Saya nggak bisa login, padahal udah pake lama"
]

X_new = tfidf.transform(test)
prediction = svm_tf_model.predict(X_new)
predicted_labels = le.inverse_transform(prediction)

for komentar, label in zip(test, predicted_labels):
    print(f"Komentar: {komentar}\nSentimen: {label}\n")

Komentar: susah login. jelek banget nih aplikasi
Sentimen: Negative

Komentar: Mantap, aplikasinya sangat membantu
Sentimen: Positive

Komentar: fitur masih kurang lengkap
Sentimen: Negative

Komentar: Saya nggak bisa login, padahal udah pake lama
Sentimen: Negative

